# Treino de um Modelo (pipeline real)

Treina pelo `TrainingService` — o MESMO caminho usado pela API e pelo
benchmark — a partir de um `.npz` (`X_train`/`y_train`). Salva o modelo
como artefato de inferência (sem estado do otimizador) com o
`input_contract` que garante paridade treino↔inferência.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "app").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

## Reprodutibilidade (semente numpy + TensorFlow)

In [ ]:
import numpy as np

try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(0)
except Exception:  # pragma: no cover
    np.random.seed(0)

## 1. Dataset sintético `.npz` (espectrograma pequeno)

In [ ]:
import tempfile
import numpy as np

rng = np.random.default_rng(0)
n, T, F = 300, 32, 16
X = rng.standard_normal((n, T, F)).astype("float32")
y = rng.integers(0, 2, n).astype("int64")
X[y == 1] += 0.6  # separa as classes → treinável em poucas épocas

workdir = Path(tempfile.mkdtemp(prefix="nb_train_"))
npz = workdir / "dataset.npz"
np.savez(npz, X_train=X, y_train=y)
print("dataset:", npz)

## 2. Treinar via TrainingService

In [ ]:
from app.core.interfaces.base import ProcessingStatus
from app.domain.services.training_service import TrainingService

models_dir = workdir / "models"
svc = TrainingService(models_dir=str(models_dir))
res = svc.train_model(
    architecture="MultiscaleCNN",
    dataset_path=str(npz),
    config={"epochs": 2, "batch_size": 32, "model_name": "nb_demo"},
)
assert res.status == ProcessingStatus.SUCCESS, res.errors
md_ = res.data
print("acurácia:", round(md_.accuracy, 4))
print("artefatos:", sorted(p.name for p in models_dir.glob("nb_demo*")))

## Notas

- O `.keras` é salvo **sem o estado do otimizador** (~3× menor, load mais
  rápido, saída idêntica) — ver `trainer.save_inference_keras`.
- O `nb_demo_config.json` carrega o `input_contract` (temperatura/EER/OOD
  calibrados) lido na inferência. Ver `notebooks/pipeline/03_inference.ipynb`.